In [2]:
import os

In [7]:
workDir="/Users/waweru/Documents/2022/IBSpy_Targets/2024-12-05_5B_Yellow_Rust/2025-07-02_Watkins_assemblies/minimapKASP/2026-AprilMarch-WGS-SNPCalling/2026-04-08_SNP-KASP-PrimerDesign"

In [8]:
os.chdir(f"{workDir}")
os.getcwd()

'/Users/waweru/Documents/2022/IBSpy_Targets/2024-12-05_5B_Yellow_Rust/2025-07-02_Watkins_assemblies/minimapKASP/2026-AprilMarch-WGS-SNPCalling/2026-04-08_SNP-KASP-PrimerDesign'

In [3]:
blastFile="5B_1_100Mb.kaspSNPsKeep_sites100bpFlanks_v2.txt"

In [4]:
!python3 filterBlastKASP-SNPs.py \
  -i {blastFile} \
  -o blast_hits.filtered.tsv \
  --detail-output blast_hits.detail.tsv \
  --locus-chr chr5B \
  --locus-start 8000000 \
  --locus-end 15000000 \
  --min-offtarget-length 180 \
  --min-offtarget-pident 90 \
  --min-bitscore-ratio 0.70 


Now we have a set of non-homeologous SNPs, but we want those where paragon is a hom_alt carrier. We use pysam to generate a summary table.

``` Examples for a biallelic SNP:

0/0/0/0/0/0 → called, not ALT carrier
1/1/1/1/1/1 → called, ALT carrier, hom_alt
0/0/0/0/0/1 → called, ALT carrier, het
0/0/1/1/1/1 → called, ALT carrier, het ```

In [5]:
input_vcf = "5B_8-15Mb_PolymarkerSelectedSNPs.vcf.gz"
output_tsv = "5B_8-15Mb_PolymarkerSelectedSNPssGenotypeSummary.tsv"

In [6]:
import pysam
import csv
import pandas as pd



vcf = pysam.VariantFile(input_vcf)

with open(output_tsv, "w", newline="") as out:
    writer = csv.writer(out, delimiter="\t")
    writer.writerow([
        "chrom", "snp_pos", "ref", "alt", "vcf_id", "qual",
        "n_called", "n_alt_carriers", "n_hom_alt", "n_het", "alt_carriers"
    ])

    for rec in vcf:
        chrom = rec.contig
        pos = rec.pos
        ref = rec.ref
        alt = rec.alts[0] if rec.alts else ""
        vcf_id = "" if rec.id in (None, ".") else rec.id
        qual = "" if rec.qual is None else rec.qual

        n_called = 0
        n_alt_carriers = 0
        n_hom_alt = 0
        n_het = 0
        alt_carriers = []

        for sample_name, sample_data in rec.samples.items():
            gt = sample_data.get("GT")

            # skip completely missing GT
            if gt is None:
                continue

            # remove missing allele placeholders
            alleles = [a for a in gt if a is not None]

            # require a fully called hexaploid genotype
            if len(alleles) != 6:
                continue

            n_called += 1

            allele_set = set(alleles)

            # for biallelic SNPs, ALT is allele index 1
            has_alt = 1 in allele_set
            all_alt = allele_set == {1}
            is_mixed_ref_alt = (0 in allele_set) and (1 in allele_set)

            if has_alt:
                n_alt_carriers += 1
                alt_carriers.append(sample_name)

            if all_alt:
                n_hom_alt += 1
            elif is_mixed_ref_alt:
                n_het += 1

        writer.writerow([
            chrom,
            pos,
            ref,
            alt,
            vcf_id,
            qual,
            n_called,
            n_alt_carriers,
            n_hom_alt,
            n_het,
            ";".join(alt_carriers)
        ])

print(f"Wrote: {output_tsv}")

# optional preview
df = pd.read_csv(output_tsv, sep="\t")
df.head()

Wrote: 5B_8-15Mb_PolymarkerSelectedSNPssGenotypeSummary.tsv


[E::idx_find_and_load] Could not retrieve index file for '5B_8-15Mb_PolymarkerSelectedSNPs.vcf.gz'


,chrom,snp_pos,ref,alt,vcf_id,qual,n_called,n_alt_carriers,n_hom_alt,n_het,alt_carriers
0,chr5B,8246063,G,A,NaN,650.343994,20,8,8,0,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...
1,chr5B,8247818,C,T,NaN,1676.180054,20,6,6,0,jagger;julius;symattis;mace;stanley;arinaLrFor
2,chr5B,8247968,A,G,NaN,1339.189941,21,9,9,0,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...
3,chr5B,8250980,T,C,NaN,1814.969971,21,9,9,0,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...
4,chr5B,8251267,C,T,NaN,1680.160034,20,6,6,0,jagger;julius;symattis;mace;stanley;arinaLrFor


Here we add columns with names of samples that are hom_alt and hets to the table

In [7]:
input_vcf = "5B_8-15Mb_PolymarkerSelectedSNPs.vcf.gz"
output_tsv = "5B_8-15Mb_PolymarkerSelectedSNPssGenotypeSummary_v2.tsv"

In [8]:
import pysam
import csv
import pandas as pd


vcf = pysam.VariantFile(input_vcf)

with open(output_tsv, "w", newline="") as out:
    writer = csv.writer(out, delimiter="\t")
    writer.writerow([
        "chrom", "snp_pos", "ref", "alt", "vcf_id", "qual",
        "n_called", "n_alt_carriers", "n_hom_alt", "n_het",
        "alt_carriers", "hom_alt_samples", "het_samples"
    ])

    for rec in vcf:
        chrom = rec.contig
        pos = rec.pos
        ref = rec.ref
        alt = rec.alts[0] if rec.alts else ""
        vcf_id = "" if rec.id in (None, ".") else rec.id
        qual = "" if rec.qual is None else rec.qual

        n_called = 0
        n_alt_carriers = 0
        n_hom_alt = 0
        n_het = 0

        alt_carriers = []
        hom_alt_samples = []
        het_samples = []

        for sample_name, sample_data in rec.samples.items():
            gt = sample_data.get("GT")

            # skip missing GT
            if gt is None:
                continue

            alleles = [a for a in gt if a is not None]

            # require fully called hexaploid genotype
            if len(alleles) != 6:
                continue

            n_called += 1

            alt_count = sum(1 for a in alleles if a == 1)
            allele_set = set(alleles)

            if alt_count > 0:
                n_alt_carriers += 1
                alt_carriers.append(sample_name)

            if alt_count == 6:
                n_hom_alt += 1
                hom_alt_samples.append(sample_name)
            elif 0 < alt_count < 6 and 0 in allele_set:
                n_het += 1
                het_samples.append(sample_name)

        writer.writerow([
            chrom,
            pos,
            ref,
            alt,
            vcf_id,
            qual,
            n_called,
            n_alt_carriers,
            n_hom_alt,
            n_het,
            ";".join(alt_carriers),
            ";".join(hom_alt_samples),
            ";".join(het_samples)
        ])

print(f"Wrote: {output_tsv}")

# optional preview
df = pd.read_csv(output_tsv, sep="\t")
df.head()

Wrote: 5B_8-15Mb_PolymarkerSelectedSNPssGenotypeSummary_v2.tsv


[E::idx_find_and_load] Could not retrieve index file for '5B_8-15Mb_PolymarkerSelectedSNPs.vcf.gz'


,chrom,snp_pos,ref,alt,vcf_id,qual,n_called,n_alt_carriers,n_hom_alt,n_het,alt_carriers,hom_alt_samples,het_samples
0,chr5B,8246063,G,A,NaN,650.343994,20,8,8,0,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,NaN
1,chr5B,8247818,C,T,NaN,1676.180054,20,6,6,0,jagger;julius;symattis;mace;stanley;arinaLrFor,jagger;julius;symattis;mace;stanley;arinaLrFor,NaN
2,chr5B,8247968,A,G,NaN,1339.189941,21,9,9,0,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,NaN
3,chr5B,8250980,T,C,NaN,1814.969971,21,9,9,0,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,NaN
4,chr5B,8251267,C,T,NaN,1680.160034,20,6,6,0,jagger;julius;symattis;mace;stanley;arinaLrFor,jagger;julius;symattis;mace;stanley;arinaLrFor,NaN


Now to slect for marker wehere the hom_alt samples includes paragon

In [9]:
import pandas as pd

df = pd.read_csv("5B_8-15Mb_PolymarkerSelectedSNPssGenotypeSummary_v2.tsv", sep="\t")

df_paragon = df[df["hom_alt_samples"].str.contains("paragon", na=False)]

df_paragon.head()

,chrom,snp_pos,ref,alt,vcf_id,qual,n_called,n_alt_carriers,n_hom_alt,n_het,alt_carriers,hom_alt_samples,het_samples
0,chr5B,8246063,G,A,NaN,650.343994,20,8,8,0,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,NaN
2,chr5B,8247968,A,G,NaN,1339.189941,21,9,9,0,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,NaN
3,chr5B,8250980,T,C,NaN,1814.969971,21,9,9,0,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,NaN
5,chr5B,8260100,T,C,NaN,954.437988,19,8,8,0,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,NaN
6,chr5B,8260299,T,A,NaN,4125.160156,21,15,14,1,paragon;jagger;norin61;cadenza;julius;WATDE018...,paragon;jagger;norin61;cadenza;julius;WATDE018...,symattis


In [10]:
df_paragon.to_csv("5B_8-15Mb_PolymarkerSelectedSNPssGenotypeSummary_v2ParagonHom.tsv", sep="\t", index=False)

now to keep the rows from polymarker file where the the SNPs are homozygous alternate with paragon

In [11]:
polymarkerPrimers="5B_8-15Mb_KeepSitesBlastFiltKaspSNPs_bracketedPolymarkerPrimer.xlsx"
paragonHomSNPs="5B_8-15Mb_PolymarkerSelectedSNPssGenotypeSummary_v2ParagonHom.xlsx"

In [14]:
dfPolymarker = pd.read_excel(polymarkerPrimers, sheet_name="1")

dfPolymarker.head()

,Marker,SNP,RegionSize,chromosome,total_contigs,contig_regions,SNP_type,A,B,common,primer_type,orientation,A_TM,B_TM,common_TM,selected_from,product_size,errors,repetitive,total_hits
0,chr5B_14997972,G51A,NaN,chr5B,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,The marker is in a repetitive region (23920 hi...,True,23920
1,chr5B_8246063,G51A,301.0,chr5B,1,chrUn:65446456-65446540,non-homoeologous,ttcacaagccaagccgacG,ttcacaagccaagccgacA,ggaggcgacaacattgacct,chromosome_nonspecific,forward,61.251,60.153,60.322,second,69.0,NaN,False,1
2,chr5B_8247197,A51C,301.0,chr5B,1,chrUn:65445335-65445403,non-homoeologous,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A(forward):considered 8; low tm 8; ok 0|common...,False,1
3,chr5B_8247818,C51T,301.0,chr5B,1,chrUn:65444750-65444844,non-homoeologous,tcaaatctgttacaaaaattcaccG,tcaaatctgttacaaaaattcaccA,cggcgatccataaaccttgg,chromosome_nonspecific,reverse,57.077,NaN,59.056,first+,78.0,NaN,False,1
4,chr5B_8247968,A51G,301.0,chr5B,1,chrUn:65444592-65444664,non-homoeologous,gtggatcctcccttccccA,gtggatcctcccttccccG,ccttgctcccggctatgg,chromosome_nonspecific,forward,60.308,61.444,59.888,first,87.0,NaN,False,1


In [18]:
len(dfPolymarker)

599

In [15]:
dfParagonHomSNPs = pd.read_excel(paragonHomSNPs)

dfParagonHomSNPs.head()

,Marker,chrom,snp_pos,ref,alt,vcf_id,qual,n_called,n_alt_carriers,n_hom_alt,n_het,alt_carriers,hom_alt_samples,het_samples
0,chr5B_8246063,chr5B,8246063,G,A,NaN,650.343994,20,8,8,0,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,NaN
1,chr5B_8247968,chr5B,8247968,A,G,NaN,1339.189941,21,9,9,0,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,NaN
2,chr5B_8250980,chr5B,8250980,T,C,NaN,1814.969971,21,9,9,0,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,NaN
3,chr5B_8260100,chr5B,8260100,T,C,NaN,954.437988,19,8,8,0,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,paragon;norin61;cadenza;WATDE0187;lancer;WATDE...,NaN
4,chr5B_8260299,chr5B,8260299,T,A,NaN,4125.160156,21,15,14,1,paragon;jagger;norin61;cadenza;julius;WATDE018...,paragon;jagger;norin61;cadenza;julius;WATDE018...,symattis


In [19]:
len(dfParagonHomSNPs)

204

In [16]:
df2_filtered = dfPolymarker[dfPolymarker["Marker"].isin(dfParagonHomSNPs["Marker"])]

In [20]:
len(df2_filtered)

204

In [17]:
df2_filtered.head()

,Marker,SNP,RegionSize,chromosome,total_contigs,contig_regions,SNP_type,A,B,common,primer_type,orientation,A_TM,B_TM,common_TM,selected_from,product_size,errors,repetitive,total_hits
1,chr5B_8246063,G51A,301.0,chr5B,1,chrUn:65446456-65446540,non-homoeologous,ttcacaagccaagccgacG,ttcacaagccaagccgacA,ggaggcgacaacattgacct,chromosome_nonspecific,forward,61.251,60.153,60.322,second,69.0,NaN,False,1
4,chr5B_8247968,A51G,301.0,chr5B,1,chrUn:65444592-65444664,non-homoeologous,gtggatcctcccttccccA,gtggatcctcccttccccG,ccttgctcccggctatgg,chromosome_nonspecific,forward,60.308,61.444,59.888,first,87.0,NaN,False,1
7,chr5B_8250980,T51C,301.0,chr5B,1,chrUn:65441588-65441688,non-homoeologous,cggagcaggggtctagatT,cggagcaggggtctagatC,gggttctggctggatgagag,chromosome_nonspecific,forward,58.172,58.653,59.820,second,56.0,NaN,False,1
10,chr5B_8260100,T51C,301.0,chr5B,1,chrUn:65432475-65432575,non-homoeologous,tagtgtgggccgggtttatT,tagtgtgggccgggtttatC,actccacaaagtcgcttgaa,chromosome_nonspecific,forward,59.007,59.457,57.674,second,74.0,NaN,False,1
12,chr5B_8260299,T51A,301.0,chr5B,1,chrUn:65432270-65432370,non-homoeologous,gagtatgtggagtgctcatcgT,gagtatgtggagtgctcatcgA,gaccaaagctatgcgtaggc,chromosome_nonspecific,forward,59.900,59.641,59.062,second,54.0,NaN,False,1


In [21]:
df2_filtered.to_excel("5B_8_15Mb_paragonHomSNPsPolyMarkerPrimers.xlsx")

keeping all the columns from both dfs.
```You want:
👉 keep all rows from df2 (your main table)
👉 add selected columns from df1 based on marker```

In [24]:
#df_merged = df2.merge(df1, on="marker", how="left")

df_merged = dfParagonHomSNPs.merge(dfPolymarker, on="Marker", how="left")

In [25]:
len(df_merged)

204

In [26]:
df_merged.to_excel("5B_8_15Mb_paragonHomSNPsPolyMarkerPrimersAllInfo.xlsx")

mpw to generate a primer ready table from the primer sequences with the HEX/FAM sequences added.

In [32]:
primerSeqsFile="5B_8_15Mb_paragonHomSNPsPolyMarkerPrimersAllInfo.xlsx"

In [33]:
# read your file if needed
# 
df = pd.read_excel(primerSeqsFile, sheet_name="4")

In [34]:
df.head()

,PrimerName,A(Ref-WATDE0042)-Hex,B(Alt-Paragon)-Fam,common
0,chr5B_8260299_T51A,GAAGGTCGGAGTCAACGGATTgagtatgtggagtgctcatcgT,GAAGGTGACCAAGTTCATGCTgagtatgtggagtgctcatcgA,gaccaaagctatgcgtaggc
1,chr5B_8327449_G51T,GAAGGTCGGAGTCAACGGATTtcaccgaacgatggcaactG,GAAGGTGACCAAGTTCATGCTtcaccgaacgatggcaactT,tcagtgcctggtggaacaat
2,chr5B_8585621_G51T,GAAGGTCGGAGTCAACGGATTgcccgagatgaaatttaatggcC,GAAGGTGACCAAGTTCATGCTgcccgagatgaaatttaatggcA,ctctttcccgcttcccagtt
3,chr5B_8586220_A51G,GAAGGTCGGAGTCAACGGATTtcgatgatggtgtcgacgA,GAAGGTGACCAAGTTCATGCTtcgatgatggtgtcgacgG,tcacatgatggagcagctcg
4,chr5B_9130702_C51T,GAAGGTCGGAGTCAACGGATTccctggatcccaaagcaagG,GAAGGTGACCAAGTTCATGCTccctggatcccaaagcaagA,gctgcatgcatttcgacagt


In [35]:
import pandas as pd



# rename columns to simpler names
df = df.rename(columns={
    "PrimerName": "marker",
    "A(Ref-WATDE0042)-Hex": "ref_hex",
    "B(Alt-Paragon)-Fam": "alt_fam",
    "common": "common"
})

# convert wide -> long
df_long = df.melt(
    id_vars="marker",
    value_vars=["ref_hex", "alt_fam", "common"],
    var_name="primer_type",
    value_name="sequence"
)

# make final primer name with suffix
suffix_map = {
    "ref_hex": "ref-hex",
    "alt_fam": "alt-fam",
    "common": "common"
}

df_long["primer_name"] = df_long["marker"] + "_" + df_long["primer_type"].map(suffix_map)

# optional: keep columns in desired order
df_long = df_long[["marker", "primer_name", "primer_type", "sequence"]]

df_long.head()

,marker,primer_name,primer_type,sequence
0,chr5B_8260299_T51A,chr5B_8260299_T51A_ref-hex,ref_hex,GAAGGTCGGAGTCAACGGATTgagtatgtggagtgctcatcgT
1,chr5B_8327449_G51T,chr5B_8327449_G51T_ref-hex,ref_hex,GAAGGTCGGAGTCAACGGATTtcaccgaacgatggcaactG
2,chr5B_8585621_G51T,chr5B_8585621_G51T_ref-hex,ref_hex,GAAGGTCGGAGTCAACGGATTgcccgagatgaaatttaatggcC
3,chr5B_8586220_A51G,chr5B_8586220_A51G_ref-hex,ref_hex,GAAGGTCGGAGTCAACGGATTtcgatgatggtgtcgacgA
4,chr5B_9130702_C51T,chr5B_9130702_C51T_ref-hex,ref_hex,GAAGGTCGGAGTCAACGGATTccctggatcccaaagcaagG


In [37]:
df = pd.read_excel(primerSeqsFile, sheet_name="4")

In [38]:
rows = []

for _, r in df.iterrows():
    marker = r["PrimerName"]

    rows.append({
        "primer_name": f"{marker}_ref-hex",
        "sequence": r["A(Ref-WATDE0042)-Hex"]
    })

    rows.append({
        "primer_name": f"{marker}_alt-fam",
        "sequence": r["B(Alt-Paragon)-Fam"]
    })

    rows.append({
        "primer_name": f"{marker}_common",
        "sequence": r["common"]
    })

df_out = pd.DataFrame(rows)

df_out.head()

,primer_name,sequence
0,chr5B_8260299_T51A_ref-hex,GAAGGTCGGAGTCAACGGATTgagtatgtggagtgctcatcgT
1,chr5B_8260299_T51A_alt-fam,GAAGGTGACCAAGTTCATGCTgagtatgtggagtgctcatcgA
2,chr5B_8260299_T51A_common,gaccaaagctatgcgtaggc
3,chr5B_8327449_G51T_ref-hex,GAAGGTCGGAGTCAACGGATTtcaccgaacgatggcaactG
4,chr5B_8327449_G51T_alt-fam,GAAGGTGACCAAGTTCATGCTtcaccgaacgatggcaactT


### July ^7^ 2026

Generate a haplotype like SNP matrix to check how similar WATDE0898 and WATDE0042 are across the locus.
This is becaaae based on the slides shared by Xiaohui, he used WATDE0898 for the comparison of sequences which may not always have the same haplotype as WATDE0042. One ws used for YR and the other for LR, but they are not the same at the locus.

In [10]:
import pysam
import pandas as pd


def allele_to_base(gt_value, alleles):
    """
    Convert VCF genotype allele index to nucleotide.
    Example:
        gt_value = 0 -> REF
        gt_value = 1 -> first ALT
    """
    if gt_value is None:
        return "N"
    if gt_value >= len(alleles):
        return "N"
    return alleles[gt_value]


def make_haplotype_table(
    vcf_path,
    chrom,
    start,
    end,
    output_tsv,
    samples=None,
    snps_only=True
):
    """
    Create a SNP nucleotide-call matrix from a VCF over a genomic interval.

    Parameters
    ----------
    vcf_path : str
        Path to bgzipped and indexed VCF, e.g. file.vcf.gz
    chrom : str
        Chromosome/contig name, e.g. "chr1" or "1"
    start : int
        1-based interval start
    end : int
        1-based interval end
    output_tsv : str
        Output TSV file
    samples : list or None
        Optional subset of samples. If None, use all samples in VCF.
    snps_only : bool
        If True, keep only SNPs, excluding indels and structural variants.
    """

    vcf = pysam.VariantFile(vcf_path)

    if samples is None:
        samples = list(vcf.header.samples)

    rows = []

    # initialise one row per sample
    sample_rows = {
        sample: {
            "sample": sample,
            "haplotype_1": "",
            "haplotype_2": "",
            "combined_genotype_string": ""
        }
        for sample in samples
    }

    snp_columns = []

    # region string avoids confusion around 0-based vs 1-based fetch coordinates
    region = f"{chrom}:{start}-{end}"

    for record in vcf.fetch(region=region):
        ref = record.ref
        alts = list(record.alts or [])
        alleles = [ref] + alts

        # Keep only simple SNPs if requested
        if snps_only:
            if len(ref) != 1:
                continue
            if any(len(alt) != 1 for alt in alts):
                continue

        snp_id = record.id if record.id else f"{record.chrom}_{record.pos}"
        colname = f"{snp_id}_{record.chrom}:{record.pos}_{ref}>{','.join(alts)}"
        snp_columns.append(colname)

        for sample in samples:
            call = record.samples[sample]
            gt = call.get("GT")

            if gt is None:
                nucleotide_call = "N/N"
                hap1_base = "N"
                hap2_base = "N"
            else:
                bases = [allele_to_base(a, alleles) for a in gt]

                # diploid expected; adjust if haploid/polyploid
                if len(bases) == 1:
                    hap1_base = bases[0]
                    hap2_base = "N"
                else:
                    hap1_base = bases[0]
                    hap2_base = bases[1]

                separator = "|" if getattr(call, "phased", False) else "/"
                nucleotide_call = separator.join(bases)

            sample_rows[sample][colname] = nucleotide_call
            sample_rows[sample]["haplotype_1"] += hap1_base
            sample_rows[sample]["haplotype_2"] += hap2_base
            sample_rows[sample]["combined_genotype_string"] += nucleotide_call + ";"

    for sample in samples:
        sample_rows[sample]["combined_genotype_string"] = sample_rows[sample][
            "combined_genotype_string"
        ].rstrip(";")
        rows.append(sample_rows[sample])

    df = pd.DataFrame(rows)

    # Put main columns first, SNP columns after
    first_cols = ["sample", "haplotype_1", "haplotype_2", "combined_genotype_string"]
    df = df[first_cols + snp_columns]

    df.to_csv(output_tsv, sep="\t", index=False)

    return df

In [13]:
input_vcf = f"{os.getcwd()}/5B_8-15Mb_PolymarkerSelectedSNPs.vcf.gz"

In [16]:
# index the vcf file

import pysam

input_vcf = f"{os.getcwd()}/5B_8-15Mb_PolymarkerSelectedSNPs.vcf.gz"

pysam.tabix_index(input_vcf, preset="vcf", force=True)

print(f"Created index: {input_vcf}.tbi")

Created index: /Users/waweru/Documents/2022/IBSpy_Targets/2024-12-05_5B_Yellow_Rust/2025-07-02_Watkins_assemblies/minimapKASP/2026-AprilMarch-WGS-SNPCalling/2026-04-08_SNP-KASP-PrimerDesign/5B_8-15Mb_PolymarkerSelectedSNPs.vcf.gz.tbi


In [18]:
df = make_haplotype_table(
    vcf_path=input_vcf,
    chrom="chr5B",
    start=8000000,
    end=15000000,
    output_tsv="haplotype_table.tsv"
)

print(df.head())

    sample                                        haplotype_1  \
0  paragon  ACGCCCATGACTAAACTCAGCCTTCTCATCGAGATGACGTTCCTTC...   
1   jagger  GTATTTATGACTAAACTCAGCCTTCTCATCGAGATGACGTTCCTTC...   
2  norin61  ACGCCCATGACTAAACTCAGCCTTCTCATCGAGATGACGTTCCTTC...   
3  cadenza  ACGCCCATGACTAAACTCAGCCTTCTCATCGAGATGACGTTCCTTC...   
4   julius  GTATTTACNNTCGGTGTTCCNCCCGNTTTGGANGACGTNTACCTTC...   

                                         haplotype_2  \
0  ACGCCCATGACTAAACTCAGCCTTCTCATCGAGATGACGTTCCTTC...   
1  GTATTTATGACTAAACTCAGCCTTCTCATCGAGATGACGTTCCTTC...   
2  ACGCCCATGACTAAACTCAGCCTTCTCATCGAGATGACGTTCCTTC...   
3  ACGCCCATGACTAAACTCAGCCTTCTCATCGAGATGACGTTCCTTC...   
4  GTATTTACNNTCGGTGTTCCNCCCGNTTTGGANGACGTNTACCTTC...   

                            combined_genotype_string  \
0  A/A/A/A/A/A;C/C/C/C/C/C;G/G/G/G/G/G;C/C/C/C/C/...   
1  G/G/G/G/G/G;T/T/T/T/T/T;A/A/A/A/A/A;T/T/T/T/T/...   
2  A/A/A/A/A/A;C/C/C/C/C/C;G/G/G/G/G/G;C/C/C/C/C/...   
3  A/A/A/A/A/A;C/C/C/C/C/C;G/G/G/G/G/G;C/C/C/C/C

Now we want SNP that are homozyous ref in just the six or so lines of intrest, one or all of them, this will be the useful ones for the haplotype calls

In [19]:
import pandas as pd
import re


def get_ref_from_snp_col(col):
    """
    Extract REF allele from SNP column names like:
        SNP1_chr1:100_T>G
        chr1_100_T>G
        chr1:100_T>G
    """
    match = re.search(r"_([ACGT]+)>", col)
    if match:
        return match.group(1)

    return None


def is_homozygous_ref(call, ref):
    """
    Check whether a genotype call is homozygous REF.

    Works for calls like:
        T/T
        T|T
        T/T/T/T/T/T
        T|T|T|T|T|T

    Returns False for:
        T/T/T/T/T/G
        G/G/G/G/G/G when REF is T
        missing calls
    """
    if pd.isna(call) or ref is None:
        return False

    call = str(call).strip()

    if call in [".", "./.", ".|.", "N", "N/N", "NA", "nan"]:
        return False

    alleles = re.split(r"[\/|]", call)
    alleles = [a for a in alleles if a not in ["", ".", "N", "NA"]]

    if len(alleles) == 0:
        return False

    return all(a == ref for a in alleles)


def find_target_specific_homozygous_ref_snps(
    haplotype_table,
    lines_of_interest,
    output_file,
    sample_col="sample",
    mode="all"
):
    """
    Find SNPs that are homozygous REF in target lines but not in the rest.

    Parameters
    ----------
    haplotype_table : str
        Path to haplotype table TSV file.

    lines_of_interest : list
        List of target sample/line names.

    output_file : str
        Output TSV file.

    sample_col : str
        Name of the sample column.

    mode : str
        "all" = SNP must be homozygous REF in all target lines.
        "any" = SNP must be homozygous REF in at least one target line.

    Returns
    -------
    pandas.DataFrame
        Filtered SNP results.
    """

    if mode not in ["all", "any"]:
        raise ValueError("mode must be either 'all' or 'any'")

    df = pd.read_csv(haplotype_table, sep="\t")

    target = df[df[sample_col].isin(lines_of_interest)].copy()
    rest = df[~df[sample_col].isin(lines_of_interest)].copy()

    missing = set(lines_of_interest) - set(target[sample_col])

    if missing:
        raise ValueError(f"These target lines were not found in the table: {missing}")

    metadata_cols = {
        sample_col,
        "population",
        "group",
        "haplotype_1",
        "haplotype_2",
        "combined_genotype_string",
        "haplotype_pair",
        "count",
        "frequency"
    }

    snp_cols = [col for col in df.columns if col not in metadata_cols]

    results = []

    for snp in snp_cols:
        ref = get_ref_from_snp_col(snp)

        if ref is None:
            print(f"Skipping column because REF could not be extracted: {snp}")
            continue

        target_hom_ref = target[snp].apply(lambda x: is_homozygous_ref(x, ref))
        rest_hom_ref = rest[snp].apply(lambda x: is_homozygous_ref(x, ref))

        if mode == "all":
            keep_snp = target_hom_ref.all() and not rest_hom_ref.any()
        else:
            keep_snp = target_hom_ref.any() and not rest_hom_ref.any()

        if keep_snp:
            results.append({
                "snp": snp,
                "ref": ref,
                "mode": mode,
                "target_homozygous_ref_count": int(target_hom_ref.sum()),
                "rest_homozygous_ref_count": int(rest_hom_ref.sum()),
                "target_homozygous_ref_lines": ";".join(
                    target.loc[target_hom_ref, sample_col].astype(str)
                ),
                "target_calls": "; ".join(
                    target[sample_col].astype(str) + "=" + target[snp].astype(str)
                ),
                "rest_calls": "; ".join(
                    rest[sample_col].astype(str) + "=" + rest[snp].astype(str)
                )
            })

    result_df = pd.DataFrame(results)
    result_df.to_csv(output_file, sep="\t", index=False)

    return result_df

In [20]:
lines_of_interest = [
    "WATDE0075",
    "WATDE0076",
    "WATDE0776",
    "WATDE0549",
    "WATDE0898",
    "WATDE0552"
]

In [21]:


result = find_target_specific_homozygous_ref_snps(
    haplotype_table="haplotype_table.tsv",
    lines_of_interest=lines_of_interest,
    output_file="target_specific_homozygous_ref_snps.tsv",
    sample_col="sample",
    mode="all"
)

print(result)

                                snp ref mode  target_homozygous_ref_count  \
0   chr5B_8260299_chr5B:8260299_T>A   T  all                            6   
1   chr5B_8267998_chr5B:8267998_G>A   G  all                            6   
2   chr5B_9134924_chr5B:9134924_T>C   T  all                            6   
3   chr5B_9148655_chr5B:9148655_C>T   C  all                            6   
4   chr5B_9222624_chr5B:9222624_A>T   A  all                            6   
5   chr5B_9246900_chr5B:9246900_A>G   A  all                            6   
6   chr5B_9248767_chr5B:9248767_C>G   C  all                            6   
7   chr5B_9260948_chr5B:9260948_A>G   A  all                            6   
8   chr5B_9264280_chr5B:9264280_T>C   T  all                            6   
9   chr5B_9264449_chr5B:9264449_C>G   C  all                            6   
10  chr5B_9266316_chr5B:9266316_G>A   G  all                            6   
11  chr5B_9300433_chr5B:9300433_G>A   G  all                            6   

In [22]:
result = find_target_specific_homozygous_ref_snps(
    haplotype_table="haplotype_table.tsv",
    lines_of_interest=lines_of_interest,
    output_file="target_specific_homozygous_ref_snps_any_target.tsv",
    sample_col="sample",
    mode="any"
)